# Libraries

In [3]:
import os, re, time, json, urllib.request
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

env_path = Path.home() / ".gpt_config.env" 
load_dotenv(env_path)
API_KEY = os.getenv("API_KEY")

if not API_KEY:
    raise ValueError("API_KEY not found in ~/.gpt_config.env")

client = OpenAI(api_key=API_KEY)

In [4]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## function calls

In [3]:
def call_api(url):
	time.sleep(1)
	url = url.replace(' ', '+')
	print(url)

	req = urllib.request.Request(url) 
	with urllib.request.urlopen(req) as response:
		call = response.read()

	return call

In [4]:
def get_prompt_header(mask):
	'''
	mask: [1/0 x 6], denotes whether each prompt component is used

	output: prompt
	'''
	url_1 = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=gene&retmax=5&retmode=json&sort=relevance&term=LMP10'
	call_1 = call_api(url_1)

	url_2 = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=gene&retmax=5&retmode=json&id=19171,5699,8138'
	call_2 = call_api(url_2)

	url_3 = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=snp&retmax=10&retmode=json&id=1217074595' 
	call_3 = call_api(url_3)

	url_4 = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=omim&retmax=20&retmode=json&sort=relevance&term=Meesmann+corneal+dystrophy'
	call_4 = call_api(url_4)

	url_5 = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=omim&retmax=20&retmode=json&id=618767,601687,300778,148043,122100'
	call_5 = call_api(url_5)

	url_6 = 'https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Put&PROGRAM=blastn&MEGABLAST=on&DATABASE=nt&FORMAT_TYPE=XML&QUERY=ATTCTGCCTTTAGTAATTTGATGACAGAGACTTCTTGGGAACCACAGCCAGGGAGCCACCCTTTACTCCACCAACAGGTGGCTTATATCCAATCTGAGAAAGAAAGAAAAAAAAAAAAGTATTTCTCT&HITLIST_SIZE=5'
	call_6 = call_api(url_6)
	rid = re.search('RID = (.*)\n', call_6.decode('utf-8')).group(1)

	url_7 = f'https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Get&FORMAT_TYPE=Text&RID={rid}'
	time.sleep(30)
	call_7 = call_api(url_7)

	prompt = ''
	prompt += 'Hello. Your task is to use NCBI Web APIs to answer genomic questions.\n'
	#prompt += 'There are two types of Web APIs you can use: Eutils and BLAST.\n\n'

	if mask[0]:
		# Doc 0 is about Eutils
		prompt += 'You can call Eutils by: "[https://eutils.ncbi.nlm.nih.gov/entrez/eutils/{esearch|efetch|esummary}.fcgi?db={gene|snp|omim}&retmax={}&{term|id}={term|id}]".\n'
		prompt += 'esearch: input is a search term and output is database id(s).\n'
		prompt += 'efectch/esummary: input is database id(s) and output is full records or summaries that contain name, chromosome location, and other information.\n'
		prompt += 'Normally, you need to first call esearch to get the database id(s) of the search term, and then call efectch/esummary to get the information with the database id(s).\n'
		prompt += 'Database: gene is for genes, snp is for SNPs, and omim is for genetic diseases.\n\n'

	if mask[1]:
		# Doc 1 is about BLAST
		prompt += 'For DNA sequences, you can use BLAST by: "[https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD={Put|Get}&PROGRAM=blastn&MEGABLAST=on&DATABASE=nt&FORMAT_TYPE={XML|Text}&QUERY={sequence}&HITLIST_SIZE={max_hit_size}]".\n'
		prompt += 'BLAST maps a specific DNA {sequence} to its chromosome location among different specices.\n'
		prompt += 'You need to first PUT the BLAST request and then GET the results using the RID returned by PUT.\n\n'

	if any(mask[2:]):
		prompt += 'Here are some examples:\n\n'

	if mask[2]:
		# Example 1 is from gene alias task 
		prompt += f'Question: What is the official gene symbol of LMP10?\n'
		prompt += f'[{url_1}]->[{call_1}]\n' 
		prompt += f'[{url_2}]->[{call_2}]\n'
		prompt += f'Answer: PSMB10\n\n'

	if mask[3]:
		# Example 2 is from SNP gene task
		prompt += f'Question: Which gene is SNP rs1217074595 associated with?\n'
		prompt += f'[{url_3}]->[{call_3}]\n'
		prompt += f'Answer: LINC01270\n\n'

	if mask[4]:
		# Example 3 is from gene disease association
		prompt += f'Question: What are genes related to Meesmann corneal dystrophy?\n'
		prompt += f'[{url_4}]->[{call_4}]\n'
		prompt += f'[{url_5}]->[{call_5}]\n'
		prompt += f'Answer: KRT12, KRT3\n\n'

	if mask[5]:
		# Example 4 is for BLAST
		prompt += f'Question: Align the DNA sequence to the human genome:ATTCTGCCTTTAGTAATTTGATGACAGAGACTTCTTGGGAACCACAGCCAGGGAGCCACCCTTTACTCCACCAACAGGTGGCTTATATCCAATCTGAGAAAGAAAGAAAAAAAAAAAAGTATTTCTCT\n'
		prompt += f'[{url_6}]->[{rid}]\n'
		prompt += f'[{url_7}]->[{call_7}]\n'
		prompt += f'Answer: chr15:91950805-91950932\n\n'

	return prompt

In [13]:
def run_GeneGPT_evaluation(
    data,
    prompt,
    save_dir,
    result_filename,
    question_col,
    gold_col,
    model,
    cut_length=100000,
    max_api_calls=10,
    api_response_limit=10000,
    answer_format_instructions=None,
    temperature=0
):
        
    # Create save directory
    os.makedirs(save_dir, exist_ok=True)
    result_path = os.path.join(save_dir, result_filename)
    
    # Default answer format instructions
    if answer_format_instructions is None:
        answer_format_instructions = (
            "Please provide your answer in the following JSON format for the Question asked:\n"
            "{\n"
            '  "answer": "<correct answer>"\n'
            "}\n"
        )
    
    # Load partial progress if exists
    if os.path.exists(result_path):
        with open(result_path) as f:
            all_results = json.load(f)
        completed_questions = {entry["question"] for entry in all_results}
        print(f"Resuming from {len(completed_questions)} completed questions.")
    else:
        all_results = []
        completed_questions = set()
        print("Starting fresh.")
    
    print(f"Total questions: {len(data)}")
    
    # Loop through all questions
    for counter, (idx, row) in enumerate(data.iterrows(), 1):
        question = row[question_col]
        gold = row[gold_col]
        
        if question in completed_questions:
            print(f"Skipping completed question {counter}: {question[:10]}")
            continue
        
        print(f"\n=== Question {counter}/{len(data)} ===")
        print("Q:", question)
        print("Gold:", gold)
        
        # Build initial prompt
        q_prompt = (
            prompt
            + f"Question: {question}\n"
            + answer_format_instructions
            + "When you decide to call a web API, always enclose the URL in square brackets [ ] exactly, "
            + "so that I can automatically fetch it. Do not use code fences.\n"
        )
        
        prompts = []
        num_calls = 0
        
        while True:
            
            if len(q_prompt) > cut_length:
                q_prompt = q_prompt[-cut_length:]
            
            # Call model
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": q_prompt}],
                    temperature=temperature,
                )
            except Exception as e:
                print("Error during model call:", e)
                break
            
            text = response.choices[0].message.content.strip()
            num_calls += 1
            prompts.append([q_prompt, text])
            
            # Detect URLs in model response
            matches = re.findall(r"\[(https?://[^\[\]]+)\]", text)
            
            if matches:
                url = matches[0]
                print("Model suggests calling:", url)
                
                try:
                    if "blast" in url and "Get" in url:
                        time.sleep(30)
                    
                    # Call the API
                    call = call_api(url)
                    
                    if "blast" in url and "Put" in url:
                        rid = re.search("RID = (.*)\n", call.decode("utf-8")).group(1)
                        call = rid
                    
                    if len(call) > api_response_limit:
                        call = call[:api_response_limit]
                    
                    # Append API result to prompt
                    q_prompt = f"{q_prompt}{text}->[{call}]\n"
                    
                except Exception as e:
                    print("Error calling API:", e)
                    break
            else:
                # No URL found - treat as final answer
                final_answer = text
                break
            
            # Safety limit on number of API calls
            if num_calls >= max_api_calls:
                final_answer = "numError"
                break
        
        print("Finished loop.")
        print("Final answer:\n", final_answer)
        
        # Save result for this question
        result = {
            "question": question,
            "gold_standard": gold,
            "model_answer": final_answer,
            "prompt_log": prompts,
        }
        all_results.append(result)
        
        # Write out cumulative results
        with open(result_path, "w") as f:
            json.dump(all_results, f, indent=2)
        
        print(f"Saved progress after question {counter} -> {result_path}")
    
    print("\nAll available questions processed or skipped.")
    return all_results

In [7]:
def parse_model_answer(answer):
    try:
        if isinstance(answer, str):
            json_match = re.search(r'\{.*\}', answer, re.DOTALL)
            if json_match:
                parsed = json.loads(json_match.group())
                if 'answer' in parsed:
                    return parsed['answer']
        return answer
    except:
        return answer

In [8]:
def check_answer_in_gold(row):
    try:
        gold_list = eval(row['gold_standard']) if isinstance(row['gold_standard'], str) else row['gold_standard']
        
        parsed_answer = row['parsed_model_answer']
        
        return parsed_answer in gold_list
    except:
        return False

In [9]:
def evaluate_results(result_file):
      
    # Load data
    with open(result_file) as f:
        results = json.load(f)
    
    df = pd.DataFrame(results)
    df = df[['question', 'gold_standard', 'model_answer']]
    
    print(f"Total questions: {len(df)}\n")
    
    # Parse and evaluate
    df['parsed_model_answer'] = df['model_answer'].apply(parse_model_answer)
    df['is_correct'] = df.apply(check_answer_in_gold, axis=1)
    
    # Calculate metrics
    accuracy = df['is_correct'].sum() / len(df) * 100
    correct_count = df['is_correct'].sum()
    total_count = len(df)
    
    # Display results
    print("="*80)
    print(f"FILE: {result_file}")
    print("="*80)
    print(f"\nAccuracy: {accuracy:.2f}% ({correct_count}/{total_count})")
    
    print("\nValue Counts:")
    print(df['is_correct'].value_counts())
    
    print("\nPercentages:")
    print(df['is_correct'].value_counts(normalize=True).mul(100).round(2))
    
    # Show incorrect cases if any
    incorrect = df[df['is_correct'] == False]
    if len(incorrect) > 0:
        print(f"\n{len(incorrect)} Incorrect answers:")
        # print(incorrect[['question', 'gold_standard', 'parsed_model_answer']].head(10))
    
    print("\n" + "="*80 + "\n")
    
    return df, accuracy

# Mechnaistic Gene

In [15]:
mech_gene = pd.read_csv("../DMDB_benchmark/Benchmarks/DMDB_mechanistic_genes_filtered.csv")
mech_gene.head(1)

,id,drug,Drug_MeshID,disease,protein,drug_name,disease_name,protein_name,protein_gene_symbol,question,count
0,['DB01219_MESH_C535694_1'],DB:DB01219,MESH:D003620,MESH:C535694,['UniProt:P21817'],Dantrolene,Malignant hyperthermia,['Ryanodine receptor 1'],['RYR1'],Which gene plays the most significant mechanistic role in how Drug Dantrolene treats or impacts the Disease Malignant hyperthermia?,1


In [16]:
mech_gene.shape

(798, 11)

## gpt-4o-mini

### GeneGPT-full

In [17]:
mech_gene_gpt4omini_genegpt_full = run_GeneGPT_evaluation(
    data = mech_gene,
    prompt = get_prompt_header([1, 1, 1, 1, 1, 1]),
    save_dir = "../data/GeneGPT_results/111111",
    result_filename = "mech_gene_gpt4omini_genegpt_full.json",
    question_col = "question",
    gold_col = "protein_gene_symbol",
    model ="gpt-4o-mini",
    cut_length=100000,
    max_api_calls=10,
    api_response_limit=10000,
    answer_format_instructions= (
        "Please provide your answer (only gene name) in the following JSON format for the Question asked:\n"
        "{\n"
        '  "answer": "<correct answer>"\n'
        "}\n"
    ),
    temperature=0)

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=gene&retmax=5&retmode=json&sort=relevance&term=LMP10
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=gene&retmax=5&retmode=json&id=19171,5699,8138
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=snp&retmax=10&retmode=json&id=1217074595
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=omim&retmax=20&retmode=json&sort=relevance&term=Meesmann+corneal+dystrophy
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=omim&retmax=20&retmode=json&id=618767,601687,300778,148043,122100
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Put&PROGRAM=blastn&MEGABLAST=on&DATABASE=nt&FORMAT_TYPE=XML&QUERY=ATTCTGCCTTTAGTAATTTGATGACAGAGACTTCTTGGGAACCACAGCCAGGGAGCCACCCTTTACTCCACCAACAGGTGGCTTATATCCAATCTGAGAAAGAAAGAAAAAAAAAAAAGTATTTCTCT&HITLIST_SIZE=5
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Get&FORMAT_TYPE=Text&RID=KJB28YNF016
Starting fresh.
Total questions: 798

=== Question 1/798 ==

### GeneGPT-slim

In [18]:
mech_gene_gpt4omini_genegpt_slim = run_GeneGPT_evaluation(
    data = mech_gene,
    prompt = get_prompt_header([0, 0, 1, 0, 0, 1]),
    save_dir = "../data/GeneGPT_results/001001",
    result_filename = "mech_gene_gpt4omini_genegpt_slim.json",
    question_col = "question",
    gold_col = "protein_gene_symbol",
    model ="gpt-4o-mini",
    cut_length=100000,
    max_api_calls=10,
    api_response_limit=10000,
    answer_format_instructions= (
        "Please provide your answer (only gene name) in the following JSON format for the Question asked:\n"
        "{\n"
        '  "answer": "<correct answer>"\n'
        "}\n"
    ),
    temperature=0)

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=gene&retmax=5&retmode=json&sort=relevance&term=LMP10
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=gene&retmax=5&retmode=json&id=19171,5699,8138
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=snp&retmax=10&retmode=json&id=1217074595
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=omim&retmax=20&retmode=json&sort=relevance&term=Meesmann+corneal+dystrophy
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=omim&retmax=20&retmode=json&id=618767,601687,300778,148043,122100
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Put&PROGRAM=blastn&MEGABLAST=on&DATABASE=nt&FORMAT_TYPE=XML&QUERY=ATTCTGCCTTTAGTAATTTGATGACAGAGACTTCTTGGGAACCACAGCCAGGGAGCCACCCTTTACTCCACCAACAGGTGGCTTATATCCAATCTGAGAAAGAAAGAAAAAAAAAAAAGTATTTCTCT&HITLIST_SIZE=5
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Get&FORMAT_TYPE=Text&RID=KJC8DS9T014
Starting fresh.
Total questions: 798

=== Question 1/798 ==

## gpt-4o

### GeneGPT-full

In [19]:
mech_gene_gpt4o_genegpt_full = run_GeneGPT_evaluation(
    data = mech_gene,
    prompt = get_prompt_header([1, 1, 1, 1, 1, 1]),
    save_dir = "../data/GeneGPT_results/111111",
    result_filename = "mech_gene_gpt4o_genegpt_full.json",
    question_col = "question",
    gold_col = "protein_gene_symbol",
    model ="gpt-4o",
    cut_length=100000,
    max_api_calls=10,
    api_response_limit=10000,
    answer_format_instructions= (
        "Please provide your answer (only gene name) in the following JSON format for the Question asked:\n"
        "{\n"
        '  "answer": "<correct answer>"\n'
        "}\n"
    ),
    temperature=0)

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=gene&retmax=5&retmode=json&sort=relevance&term=LMP10
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=gene&retmax=5&retmode=json&id=19171,5699,8138
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=snp&retmax=10&retmode=json&id=1217074595
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=omim&retmax=20&retmode=json&sort=relevance&term=Meesmann+corneal+dystrophy
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=omim&retmax=20&retmode=json&id=618767,601687,300778,148043,122100
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Put&PROGRAM=blastn&MEGABLAST=on&DATABASE=nt&FORMAT_TYPE=XML&QUERY=ATTCTGCCTTTAGTAATTTGATGACAGAGACTTCTTGGGAACCACAGCCAGGGAGCCACCCTTTACTCCACCAACAGGTGGCTTATATCCAATCTGAGAAAGAAAGAAAAAAAAAAAAGTATTTCTCT&HITLIST_SIZE=5
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Get&FORMAT_TYPE=Text&RID=KJCXJAXT014
Starting fresh.
Total questions: 798

=== Question 1/798 ==

### GeneGPT-slim

In [22]:
mech_gene_gpt4o_genegpt_slim = run_GeneGPT_evaluation(
    data = mech_gene,
    prompt = get_prompt_header([0, 0, 1, 0, 0, 1]),
    save_dir = "../data/GeneGPT_results/001001",
    result_filename = "mech_gene_gpt4o_genegpt_slim.json",
    question_col = "question",
    gold_col = "protein_gene_symbol",
    model ="gpt-4o",
    cut_length=100000,
    max_api_calls=10,
    api_response_limit=10000,
    answer_format_instructions= (
        "Please provide your answer (only gene name) in the following JSON format for the Question asked:\n"
        "{\n"
        '  "answer": "<correct answer>"\n'
        "}\n"
    ),
    temperature=0)

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=gene&retmax=5&retmode=json&sort=relevance&term=LMP10
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=gene&retmax=5&retmode=json&id=19171,5699,8138
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=snp&retmax=10&retmode=json&id=1217074595
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=omim&retmax=20&retmode=json&sort=relevance&term=Meesmann+corneal+dystrophy
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=omim&retmax=20&retmode=json&id=618767,601687,300778,148043,122100
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Put&PROGRAM=blastn&MEGABLAST=on&DATABASE=nt&FORMAT_TYPE=XML&QUERY=ATTCTGCCTTTAGTAATTTGATGACAGAGACTTCTTGGGAACCACAGCCAGGGAGCCACCCTTTACTCCACCAACAGGTGGCTTATATCCAATCTGAGAAAGAAAGAAAAAAAAAAAAGTATTTCTCT&HITLIST_SIZE=5
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Get&FORMAT_TYPE=Text&RID=KK47DXCT014
Resuming from 740 completed questions.
Total questions: 798

In [27]:
df, acc = evaluate_results("../data/GeneGPT_results/001001/mech_gene_gpt4omini_genegpt_slim.json")

Total questions: 798

FILE: ../data/GeneGPT_results/001001/mech_gene_gpt4omini_genegpt_slim.json

Accuracy: 45.24% (361/798)

Value Counts:
is_correct
False    437
True     361
Name: count, dtype: int64

Percentages:
is_correct
False    54.76
True     45.24
Name: proportion, dtype: float64

437 Incorrect answers:




In [28]:
files = [
    "../data/GeneGPT_results/111111/mech_gene_gpt4omini_genegpt_full.json",
    "../data/GeneGPT_results/111111/mech_gene_gpt4o_genegpt_full.json",
    "../data/GeneGPT_results/001001/mech_gene_gpt4omini_genegpt_slim.json",
    "../data/GeneGPT_results/001001/mech_gene_gpt4o_genegpt_slim.json"
]

results_summary = {}
for file in files:
    df, acc = evaluate_results(file)
    results_summary[file] = acc

# Summary comparison
print("\nSUMMARY COMPARISON:")
for file, acc in results_summary.items():
    print(f"{file}: {acc:.2f}%")

Total questions: 798

FILE: ../data/GeneGPT_results/111111/mech_gene_gpt4omini_genegpt_full.json

Accuracy: 37.22% (297/798)

Value Counts:
is_correct
False    501
True     297
Name: count, dtype: int64

Percentages:
is_correct
False    62.78
True     37.22
Name: proportion, dtype: float64

501 Incorrect answers:


Total questions: 798

FILE: ../data/GeneGPT_results/111111/mech_gene_gpt4o_genegpt_full.json

Accuracy: 42.73% (341/798)

Value Counts:
is_correct
False    457
True     341
Name: count, dtype: int64

Percentages:
is_correct
False    57.27
True     42.73
Name: proportion, dtype: float64

457 Incorrect answers:


Total questions: 798

FILE: ../data/GeneGPT_results/001001/mech_gene_gpt4omini_genegpt_slim.json

Accuracy: 45.24% (361/798)

Value Counts:
is_correct
False    437
True     361
Name: count, dtype: int64

Percentages:
is_correct
False    54.76
True     45.24
Name: proportion, dtype: float64

437 Incorrect answers:


Total questions: 798

FILE: ../data/GeneGPT_results/0

# GeneTuring: gene-dis-association

In [7]:
geneturing_data = pd.read_csv("../data/geneTuring/Q&A_dataset.csv")
geneturing_data.columns

Index(['Model', 'Module', 'Question', 'Goldstandard', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7'],
      dtype='object')

In [8]:
gene_dis_data = geneturing_data[geneturing_data["Module"].str.strip() == "Gene disease association"]
gene_dis_data.shape

(100, 8)

In [9]:
gene_dis_data = gene_dis_data.iloc[:, :-4]

In [10]:
gene_dis_data['Question'] = gene_dis_data['Question'].str.replace(
    r'^The name of the gene related to (.+) is$',
    r'What is the name of the gene related to \1?',
    regex=True
)

In [11]:
gene_dis_data.head(2)

,Model,Module,Question,Goldstandard
300,GeneGPT,Gene disease association,What is the name of the gene related to Hemolytic anemia due to phosphofructokinase deficiency?,PFKL
301,GeneGPT,Gene disease association,What is the name of the gene related to Distal renal tubular acidosis?,"SLC4A1, ATP6V0A4"


In [12]:
gene_dis_data['num_answers'] = gene_dis_data['Goldstandard'].apply(
    lambda x: len(x.split(',')) if isinstance(x, str) else 1
)

print("Distribution of number of answers in Goldstandard:")
print(gene_dis_data['num_answers'].value_counts().sort_index())

Distribution of number of answers in Goldstandard:
num_answers
1    66
2    18
3     8
4     3
5     1
6     2
7     1
8     1
Name: count, dtype: int64


In [29]:
gene_dis_data.shape

(100, 5)

## gpt-4o-mini

### GeneGPT-full

In [34]:
GT_gene_dis_gpt4omini_genegpt_full = run_GeneGPT_evaluation(
    data = gene_dis_data,
    prompt = get_prompt_header([1, 1, 1, 1, 1, 1]),
    save_dir = "../data/GeneGPT_results/111111",
    result_filename = "GeneTuring_gene_dis_gpt4omini_genegpt_full.json",
    question_col = "Question",
    gold_col = "Goldstandard",
    model ="gpt-4o-mini",
    cut_length=100000,
    max_api_calls=10,
    api_response_limit=10000,
    answer_format_instructions= (
        "Please provide your answer (only gene symbol) in the following JSON format for the Question asked:\n"
        "{\n"
        '  "answer": "<correct answer>"\n'
        "}\n"
    ),
    temperature=0)

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=gene&retmax=5&retmode=json&sort=relevance&term=LMP10
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=gene&retmax=5&retmode=json&id=19171,5699,8138
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=snp&retmax=10&retmode=json&id=1217074595
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=omim&retmax=20&retmode=json&sort=relevance&term=Meesmann+corneal+dystrophy
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=omim&retmax=20&retmode=json&id=618767,601687,300778,148043,122100
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Put&PROGRAM=blastn&MEGABLAST=on&DATABASE=nt&FORMAT_TYPE=XML&QUERY=ATTCTGCCTTTAGTAATTTGATGACAGAGACTTCTTGGGAACCACAGCCAGGGAGCCACCCTTTACTCCACCAACAGGTGGCTTATATCCAATCTGAGAAAGAAAGAAAAAAAAAAAAGTATTTCTCT&HITLIST_SIZE=5
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Get&FORMAT_TYPE=Text&RID=KKAFH7TK014
Starting fresh.
Total questions: 100

=== Question 1/100 ==

### GeneGPT-slim

In [35]:
GT_gene_dis_gpt4omini_genegpt_slim = run_GeneGPT_evaluation(
    data = gene_dis_data,
    prompt = get_prompt_header([0, 0, 1, 0, 0, 1]),
    save_dir = "../data/GeneGPT_results/001001",
    result_filename = "GeneTuring_gene_dis_gpt4omini_genegpt_slim.json",
    question_col = "Question",
    gold_col = "Goldstandard",
    model ="gpt-4o-mini",
    cut_length=100000,
    max_api_calls=10,
    api_response_limit=10000,
    answer_format_instructions= (
        "Please provide your answer (only gene symbol) in the following JSON format for the Question asked:\n"
        "{\n"
        '  "answer": "<correct answer>"\n'
        "}\n"
    ),
    temperature=0)

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=gene&retmax=5&retmode=json&sort=relevance&term=LMP10
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=gene&retmax=5&retmode=json&id=19171,5699,8138
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=snp&retmax=10&retmode=json&id=1217074595
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=omim&retmax=20&retmode=json&sort=relevance&term=Meesmann+corneal+dystrophy
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=omim&retmax=20&retmode=json&id=618767,601687,300778,148043,122100
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Put&PROGRAM=blastn&MEGABLAST=on&DATABASE=nt&FORMAT_TYPE=XML&QUERY=ATTCTGCCTTTAGTAATTTGATGACAGAGACTTCTTGGGAACCACAGCCAGGGAGCCACCCTTTACTCCACCAACAGGTGGCTTATATCCAATCTGAGAAAGAAAGAAAAAAAAAAAAGTATTTCTCT&HITLIST_SIZE=5
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Get&FORMAT_TYPE=Text&RID=KKAKHFZ0016
Starting fresh.
Total questions: 100

=== Question 1/100 ==

## gpt-4o

### GeneGPT-full

In [36]:
GT_gene_dis_gpt4o_genegpt_full = run_GeneGPT_evaluation(
    data = gene_dis_data,
    prompt = get_prompt_header([1, 1, 1, 1, 1, 1]),
    save_dir = "../data/GeneGPT_results/111111",
    result_filename = "GeneTuring_gene_dis_gpt4o_genegpt_full.json",
    question_col = "Question",
    gold_col = "Goldstandard",
    model ="gpt-4o",
    cut_length=100000,
    max_api_calls=10,
    api_response_limit=10000,
    answer_format_instructions= (
        "Please provide your answer (only gene symbol) in the following JSON format for the Question asked:\n"
        "{\n"
        '  "answer": "<correct answer>"\n'
        "}\n"
    ),
    temperature=0)

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=gene&retmax=5&retmode=json&sort=relevance&term=LMP10
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=gene&retmax=5&retmode=json&id=19171,5699,8138
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=snp&retmax=10&retmode=json&id=1217074595
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=omim&retmax=20&retmode=json&sort=relevance&term=Meesmann+corneal+dystrophy
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=omim&retmax=20&retmode=json&id=618767,601687,300778,148043,122100
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Put&PROGRAM=blastn&MEGABLAST=on&DATABASE=nt&FORMAT_TYPE=XML&QUERY=ATTCTGCCTTTAGTAATTTGATGACAGAGACTTCTTGGGAACCACAGCCAGGGAGCCACCCTTTACTCCACCAACAGGTGGCTTATATCCAATCTGAGAAAGAAAGAAAAAAAAAAAAGTATTTCTCT&HITLIST_SIZE=5
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Get&FORMAT_TYPE=Text&RID=KKAUGR44016
Starting fresh.
Total questions: 100

=== Question 1/100 ==

### GeneGPT-slim

In [37]:
GT_gene_dis_gpt4o_genegpt_slim = run_GeneGPT_evaluation(
    data = gene_dis_data,
    prompt = get_prompt_header([0, 0, 1, 0, 0, 1]),
    save_dir = "../data/GeneGPT_results/001001",
    result_filename = "GeneTuring_gene_dis_gpt4o_genegpt_slim.json",
    question_col = "Question",
    gold_col = "Goldstandard",
    model ="gpt-4o",
    cut_length=100000,
    max_api_calls=10,
    api_response_limit=10000,
    answer_format_instructions= (
        "Please provide your answer (only gene symbol) in the following JSON format for the Question asked:\n"
        "{\n"
        '  "answer": "<correct answer>"\n'
        "}\n"
    ),
    temperature=0)

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=gene&retmax=5&retmode=json&sort=relevance&term=LMP10
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=gene&retmax=5&retmode=json&id=19171,5699,8138
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=snp&retmax=10&retmode=json&id=1217074595
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=omim&retmax=20&retmode=json&sort=relevance&term=Meesmann+corneal+dystrophy
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=omim&retmax=20&retmode=json&id=618767,601687,300778,148043,122100
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Put&PROGRAM=blastn&MEGABLAST=on&DATABASE=nt&FORMAT_TYPE=XML&QUERY=ATTCTGCCTTTAGTAATTTGATGACAGAGACTTCTTGGGAACCACAGCCAGGGAGCCACCCTTTACTCCACCAACAGGTGGCTTATATCCAATCTGAGAAAGAAAGAAAAAAAAAAAAGTATTTCTCT&HITLIST_SIZE=5
https://blast.ncbi.nlm.nih.gov/blast/Blast.cgi?CMD=Get&FORMAT_TYPE=Text&RID=KKC198EF016
Starting fresh.
Total questions: 100

=== Question 1/100 ==

In [49]:
def evaluate_GT_results(result_file):
    """
    Evaluate GeneGPT results from a JSON file
    Handles multiple gold standard answers (comma-separated)
    """
    # Load data
    with open(result_file) as f:
        results = json.load(f)
    
    df = pd.DataFrame(results)
    df = df[['question', 'gold_standard', 'model_answer']]
    
    print(f"Total questions: {len(df)}\n")
    
    # Parse model answer (extract from JSON)
    df['parsed_model_answer'] = df['model_answer'].apply(parse_model_answer)
    
    # Split gold standard by comma and strip whitespace
    df['gold_standard_list'] = df['gold_standard'].apply(
        lambda x: [gene.strip() for gene in str(x).split(',')]
    )
    
    # Check if parsed answer matches any of the gold standard answers
    def check_answer_flexible(row):
        if pd.isna(row['parsed_model_answer']) or row['parsed_model_answer'] == '':
            return False
        
        model_ans = str(row['parsed_model_answer']).strip().upper()
        gold_list = [g.strip().upper() for g in row['gold_standard_list']]
        
        # Check if model answer matches any gold standard
        return model_ans in gold_list
    
    df['is_correct'] = df.apply(check_answer_flexible, axis=1)
    
    # Calculate metrics
    accuracy = df['is_correct'].sum() / len(df) * 100
    correct_count = df['is_correct'].sum()
    total_count = len(df)
    
    # Display results
    print("="*80)
    print(f"FILE: {result_file}")
    print("="*80)
    print(f"\nAccuracy: {accuracy:.2f}% ({correct_count}/{total_count})")
    
    print("\nValue Counts:")
    print(df['is_correct'].value_counts())
    
    print("\nPercentages:")
    print(df['is_correct'].value_counts(normalize=True).mul(100).round(2))
    
    # Show incorrect cases if any
    incorrect = df[df['is_correct'] == False]
    # if len(incorrect) > 0:
    #     print(f"\n{len(incorrect)} Incorrect answers (showing first 10):")
    #     display_df = incorrect[['question', 'gold_standard', 'parsed_model_answer']].head(10)
    #     for idx, row in display_df.iterrows():
    #         print(f"\nQ: {row['question']}")
    #         print(f"Gold: {row['gold_standard']}")
    #         print(f"Model: {row['parsed_model_answer']}")
    
    print("\n" + "="*80 + "\n")
    
    return df, accuracy

In [50]:
df, acc = evaluate_GT_results("../data/GeneGPT_results/001001/GeneTuring_gene_dis_gpt4omini_genegpt_slim.json")

Total questions: 100

FILE: ../data/GeneGPT_results/001001/GeneTuring_gene_dis_gpt4omini_genegpt_slim.json

Accuracy: 32.00% (32/100)

Value Counts:
is_correct
False    68
True     32
Name: count, dtype: int64

Percentages:
is_correct
False    68.0
True     32.0
Name: proportion, dtype: float64




In [66]:
files = [
    "../data/GeneGPT_results/111111/GeneTuring_gene_dis_gpt4omini_genegpt_full.json",
    "../data/GeneGPT_results/111111/GeneTuring_gene_dis_gpt4o_genegpt_full.json",
    "../data/GeneGPT_results/001001/GeneTuring_gene_dis_gpt4omini_genegpt_slim.json",
    "../data/GeneGPT_results/001001/GeneTuring_gene_dis_gpt4o_genegpt_slim.json"
]

results_summary = {}
for file in files:
    df, acc = evaluate_GT_results(file)
    results_summary[file] = acc

# Summary comparison
print("\nSUMMARY COMPARISON:")
for file, acc in results_summary.items():
    print(f"{file}: {acc:.2f}%")

Total questions: 100

FILE: ../data/GeneGPT_results/111111/GeneTuring_gene_dis_gpt4omini_genegpt_full.json

Accuracy: 29.00% (29/100)

Value Counts:
is_correct
False    71
True     29
Name: count, dtype: int64

Percentages:
is_correct
False    71.0
True     29.0
Name: proportion, dtype: float64


Total questions: 100

FILE: ../data/GeneGPT_results/111111/GeneTuring_gene_dis_gpt4o_genegpt_full.json

Accuracy: 59.00% (59/100)

Value Counts:
is_correct
True     59
False    41
Name: count, dtype: int64

Percentages:
is_correct
True     59.0
False    41.0
Name: proportion, dtype: float64


Total questions: 100

FILE: ../data/GeneGPT_results/001001/GeneTuring_gene_dis_gpt4omini_genegpt_slim.json

Accuracy: 32.00% (32/100)

Value Counts:
is_correct
False    68
True     32
Name: count, dtype: int64

Percentages:
is_correct
False    68.0
True     32.0
Name: proportion, dtype: float64


Total questions: 100

FILE: ../data/GeneGPT_results/001001/GeneTuring_gene_dis_gpt4o_genegpt_slim.json

Accura